# Connect to API server

In [12]:
"""
loadConfigFile.py:

   Tested with two back-2-back Ixia ports

   - Connect to the API server
   - Configure license server IP
   - Load a saved config file
   - Configure license server IP
   - Optional: Assign ports or use the ports that are in the saved config file.
   - Start all protocols
   - Verify all protocols
   - Start traffic
   - Get Traffic Item
   - Get Flow Statistics stats

Supports IxNetwork API servers:
   - Windows, Windows Connection Mgr and Linux

Requirements
   - IxNetwork 8.50
   - Python 2.7 and 3+
   - pip install requests
   - pip install ixnetwork_restpy (minimum version 1.0.51)

RestPy Doc:
    https://www.openixia.github.io/ixnetwork_restpy

Usage:
   - Enter: python <script>
"""

import json, sys, os, traceback

# Import the RestPy module
from ixnetwork_restpy import SessionAssistant, Files

# For linux and connection_manager only. Set to True to leave the session alive for debugging.
debugMode = False


# LogLevel: none, info, warning, request, request_response, all
session = SessionAssistant(IpAddress='10.36.94.225', RestPort=None, UserName='seunyang', Password='seunyang',
                            SessionName=None, SessionId=None, ApiKey=None,
                            ClearConfig=True, LogLevel='info', LogFilename='restpy.log')

ixNetwork = session.Ixnetwork
ixNetwork


2026-03-24 17:50:47 [ixnetwork_restpy.connection tid:124993348860992] [WARNING] Verification of certificates is disabled
2026-03-24 17:50:47 [ixnetwork_restpy.connection tid:124993348860992] [INFO] Determining the platform and rest_port using the 10.36.94.225 address...
2026-03-24 17:50:47 [ixnetwork_restpy.connection tid:124993348860992] [WARNING] Unable to connect to http://10.36.94.225:443.
2026-03-24 17:50:47 [ixnetwork_restpy.connection tid:124993348860992] [INFO] Connection established to `https://10.36.94.225:443 on linux`
2026-03-24 17:51:21 [ixnetwork_restpy.connection tid:124993348860992] [INFO] Using IxNetwork api server version 11.10.2508.10
2026-03-24 17:51:21 [ixnetwork_restpy.connection tid:124993348860992] [INFO] User info IxNetwork/ixnetworkweb/seunyang-77-3153576


# Loading the Configuration File: bgp_ngpf_8.30.ixncfg

In [13]:

ixNetwork.info('Loading config file: bgp_ngpf_8.30.ixncfg')
ixNetwork.LoadConfig(Files('bgp_ngpf_8.30.ixncfg', local_file=True))


2026-03-24 17:51:23 [ixnetwork_restpy.connection tid:124993348860992] [INFO] Loading config file: bgp_ngpf_8.30.ixncfg


# vport map

In [14]:

# Assign ports. Map physical ports to the configured vports.
portMap = session.PortMapAssistant()
# For the portName, get it from the loaded configuration
portMap.Map(IpAddress='10.36.88.110', CardId=1, PortId=3, Name=ixNetwork.Vport.find()[0].Name)
portMap.Map(IpAddress='10.36.88.110', CardId=1, PortId=4, Name=ixNetwork.Vport.find()[1].Name)
portMap.Connect(ForceOwnership=True)
portMap


2026-03-24 17:51:29 [ixnetwork_restpy.connection tid:124993348860992] [INFO] Adding test port hosts [10.36.88.110]...
2026-03-24 17:51:29 [ixnetwork_restpy.connection tid:124993348860992] [INFO] [10.36.88.110] is already added.
2026-03-24 17:51:33 [ixnetwork_restpy.connection tid:124993348860992] [INFO] PortMapAssistant._add_hosts duration: 3.60093092918396secs
2026-03-24 17:51:33 [ixnetwork_restpy.connection tid:124993348860992] [INFO] Connecting virtual ports to test ports using location
2026-03-24 17:51:47 [ixnetwork_restpy.connection tid:124993348860992] [INFO] PortMapAssistant._connect_ports duration: 14.612550020217896secs
2026-03-24 17:51:47 [ixnetwork_restpy.connection tid:124993348860992] [INFO] Checking virtual port link states
2026-03-24 17:51:58 [ixnetwork_restpy.connection tid:124993348860992] [INFO] PortMapAssistant._check_link_state duration: 10.306494951248169secs


# Start All Protocols

In [15]:

ixNetwork.info('Starting NGPF protocols')
ixNetwork.StartAllProtocols(Arg1='sync')

ixNetwork.info('Verify protocol sessions')
protocolSummary = session.StatViewAssistant('Protocols Summary')
protocolSummary.CheckCondition('Sessions Not Started', protocolSummary.EQUAL, 0)
protocolSummary.CheckCondition('Sessions Down', protocolSummary.EQUAL, 0)
protocolSummary


2026-03-24 17:51:58 [ixnetwork_restpy.connection tid:124993348860992] [INFO] Starting NGPF protocols
2026-03-24 17:52:13 [ixnetwork_restpy.connection tid:124993348860992] [INFO] Verify protocol sessions


# Traffic Items

In [16]:

# Get the Traffic Item name for getting Traffic Item statistics.
trafficItem = ixNetwork.Traffic.TrafficItem.find()[0]

trafficItem.Generate()

ixNetwork.info('Applying traffic')
ixNetwork.Traffic.Apply()

ixNetwork.info('Starting traffic')
ixNetwork.Traffic.StartStatelessTrafficBlocking()

trafficItemStatistics = session.StatViewAssistant('Traffic Item Statistics')
ixNetwork.info('{}\n'.format(trafficItemStatistics))

    # Get the statistic values
txFrames = trafficItemStatistics.Rows[0]['Tx Frames']
rxFrames = trafficItemStatistics.Rows[0]['Rx Frames']
ixNetwork.info('\nTraffic Item Stats:\n\tTxFrames: {}  RxFrames: {}\n'.format(txFrames, rxFrames))








2026-03-24 17:52:24 [ixnetwork_restpy.connection tid:124993348860992] [INFO] Applying traffic
2026-03-24 17:52:29 [ixnetwork_restpy.connection tid:124993348860992] [INFO] Starting traffic
2026-03-24 17:52:47 [ixnetwork_restpy.connection tid:124993348860992] [INFO] Row:0  View:Traffic Item Statistics  Sampled:2026-03-24 17:52:47.822674 UTC
	Traffic Item: BGP Traffic
	Tx Frames: 10000
	Rx Frames: 10000
	Frames Delta: 0
	Loss %: 0.000
	Tx Frame Rate: 0.000
	Rx Frame Rate: 0.000
	Tx L1 Rate (bps): 0.000
	Rx L1 Rate (bps): 0.000
	Rx Bytes: 1280000
	Tx Rate (Bps): 0.000
	Rx Rate (Bps): 0.000
	Tx Rate (bps): 0.000
	Rx Rate (bps): 0.000
	Tx Rate (Kbps): 0.000
	Rx Rate (Kbps): 0.000
	Tx Rate (Mbps): 0.000
	Rx Rate (Mbps): 0.000
	Store-Forward Avg Latency (ns): 0
	Store-Forward Min Latency (ns): 0
	Store-Forward Max Latency (ns): 0
	First TimeStamp: 00:00:01.088
	Last TimeStamp: 00:00:01.090

2026-03-24 17:52:50 [ixnetwork_restpy.connection tid:124993348860992] [INFO] 
Traffic Item Stats:
	TxFra

# Stop the traffic

In [17]:
ixNetwork.info('Stopping traffic')
ixNetwork.Traffic.StopStatelessTrafficBlocking()

2026-03-24 17:52:50 [ixnetwork_restpy.connection tid:124993348860992] [INFO] Stopping traffic


# Close the session

In [18]:
session.Session.remove()